# 🌟 W2: F&B 리뷰 감성분석 & 자동답글
**hexa-2 — Week 2** | ⏱️ 60분 | 🎯 리뷰 분류 + 저점수 자동답글

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, pandas as pd
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ 준비 완료')


## Step 1: 가게 정보 입력 (✏️)

In [ ]:
INFO={'name':'✏️ [가게명]','platform':'✏️ [배달의민족|쿠팡이츠]'}
print('✅',INFO['name'])


## Step 2: 리뷰 데이터 로드

In [ ]:
import io, requests
URL='https://raw.githubusercontent.com/Reasonofmoon/hexa-2/main/shared/fnb_reviews_sample.csv'
try:
    rdf=pd.read_csv(URL,encoding='utf-8-sig'); print(f'✅ GitHub 로드: {len(rdf)}행')
except:
    try:
        from google.colab import files; up=files.upload()
        if up: rdf=pd.read_csv(io.BytesIO(list(up.values())[0]),encoding='utf-8-sig')
        else: raise Exception('취소됨')
    except:
        rdf=pd.DataFrame({'날짜':['2026-01-01']*4,'메뉴명':['김치찌개']*4,'별점':[5,4,2,1],'리뷰내용':['완전 맛있어요!','괜찮아요','별로였어요','다시는 안 와요']})
        print('📋 기본 샘플 데이터 사용')


## Step 3: 감성 분류 (Gemini)

In [ ]:
def classify(review, rating):
    p=f"""리뷰: '{review}' (별점:{rating})
감성을 분류하세요: 긍정/부정/중립 중 하나만 출력. 다른 말 없이."""
    return model.generate_content(p).text.strip()[:4]
rdf['감성']=[classify(r['리뷰내용'],r['별점']) for _,r in rdf.iterrows()]
print(rdf[['메뉴명','별점','감성','리뷰내용']].to_string(index=False))


## Step 4: 저점수 자동답글 생성 (2점 이하)

In [ ]:
low=rdf[rdf['별점']<=2].copy()
def gen_reply(row):
    p=f"""음식점 사장님 입장에서 아래 리뷰에 공감하고 개선을 약속하는 정중한 답글을 80자 이내로:
'{row['리뷰내용']}'(별점:{row['별점']})"""
    return model.generate_content(p).text.strip()
low['답글']=[gen_reply(r) for _,r in low.iterrows()]
for _,r in low.iterrows(): print(f"[{r['별점']}★] {r['리뷰내용']}\n→ {r['답글']}\n")


## Step 5: 별점 분포 차트 + 다운로드

In [ ]:
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4))
rdf['별점'].value_counts().sort_index().plot(kind='bar',ax=ax1,color='steelblue')
ax1.set_title(f'{INFO["name"]} 별점 분포'); ax1.set_xlabel('별점'); ax1.set_ylabel('건수')
rdf['감성'].value_counts().plot(kind='pie',ax=ax2,autopct='%1.0f%%',startangle=90)
ax2.set_title('감성 비율')
plt.tight_layout(); plt.savefig('review_chart.png',dpi=150,bbox_inches='tight'); plt.show()
rdf.to_csv('review_result.csv',index=False,encoding='utf-8-sig')
if low is not None and len(low)>0: low.to_csv('auto_reply.csv',index=False,encoding='utf-8-sig')
from google.colab import files
files.download('review_result.csv'); files.download('review_chart.png')
print('✅ 완료!')
